# Stats Notebook – شرح الخوارزميات والأمثلة

يحتوي هذا النوتبوك على:
1. التوزيع المنتظم Uniform(10,20)
2. التوزيع الأسي Exponential
3. التوزيع الثنائي Binomial (مثال MCQ)
4. التوزيع الطبيعي Normal (مثال المشاهدين)
5. تجربة CLT باستخدام الأرقام 0..9
6. تقدير Gaussian MLE (μ و σ²)

## الخلية 1 – الاستيراد والإعداد العام + دالة normal_cdf

In [ ]:
import math                  # مكتبة الرياضيات (sqrt, exp, erf, factorial, ...)
import numpy as np           # مكتبة المصفوفات والحسابات العددية
import matplotlib.pyplot as plt  # مكتبة الرسم (Histogram, Curves, ...)

# ننشئ مولّد أرقام عشوائية (Random Number Generator)
# نستخدم seed ثابت حتى كل تشغيل يعطينا نفس "العشوائية"
rng = np.random.default_rng(seed=42)


def normal_cdf(x, mu=0.0, sigma=1.0):
    """
    CDF of Normal(mu, sigma^2) using math.erf

    شرح الدالة:
    - بتحسب دالة التوزيع التراكمي (CDF) لتوزيع طبيعي N(mu, sigma^2).
    - الفكرة: نحول x إلى Z-score، وبعدين نستخدم دالة erf لتقريب CDF.

    الصيغة:
    Z = (x - mu) / (sigma * sqrt(2))
    CDF(x) = 0.5 * (1 + erf(Z))
    """
    z = (x - mu) / (sigma * math.sqrt(2))
    return 0.5 * (1 + math.erf(z))


## الخلية 2 – مثال التوزيع المنتظم Uniform(10, 20)

In [ ]:
# حدود التوزيع المنتظم: أقل وأعلى زمن انتظار
a, b = 10, 20  # بالدقائق

# احتمال الانتظار على الأقل 15 دقيقة:
# في التوزيع المنتظم P(c <= X <= d) = (d - c) / (b - a)
# هنا c = 15, d = 20, b - a = 10
p_at_least_15 = (b - 15) / (b - a)

# المتوسط في U(a,b) هو منتصف المدى
mu_uniform = (a + b) / 2

# الانحراف المعياري في U(a,b) هو (b - a) / sqrt(12)
sigma_uniform = (b - a) / math.sqrt(12)

print("=== Uniform(10,20) Example ===")
print("P(X >= 15) =", p_at_least_15)
print("Mean       =", mu_uniform)
print("Std dev    =", sigma_uniform)


## الخلية 3 – مثال التوزيع الأسي Exponential (زمن إصلاح آلة)

In [ ]:
# المتوسط المعطى لوقت الإصلاح (بالساعات)
mean_time = 0.5  # μ = 0.5 ساعة

# في التوزيع الأسي: μ = 1 / λ  ⇒  λ = 1 / μ
lam = 1 / mean_time  # معدل الأحداث في الساعة

# الزمن الذي نريد حساب احتمال تجاوزه
t = 2.0  # ساعتان

# قانون التوزيع الأسي: P(X > t) = e^(-λ t)
p_more_than_2 = math.exp(-lam * t)

print("\n=== Exponential Example ===")
print("lambda (λ)      =", lam)
print("P(X > 2 hours)  =", p_more_than_2)


## الخلية 4 – مثال Binomial (MCQ test) – 3 أسئلة و4 خيارات

In [ ]:
# عدد الأسئلة
n = 3

# احتمال الإجابة الصحيحة في سؤال واحد (1 من 4)
p = 0.25


def comb(n, k):
    """
    C(n,k) = n! / (k! (n-k)!)
    دالة التوافيق، نستخدم factorial من math.
    """
    return math.factorial(n) // (math.factorial(k) * math.factorial(n - k))


def binom_pmf(k, n, p):
    """
    P(X = k) في توزيع Binomial(n, p):

    P(X = k) = C(n,k) * p^k * (1-p)^(n-k)
    حيث:
    - X = عدد الإجابات الصحيحة
    - n = عدد الأسئلة
    - p = احتمال الإجابة الصحيحة في سؤال واحد
    """
    return comb(n, k) * (p ** k) * ((1 - p) ** (n - k))


# نطبق الدالة للمثال:
# احتمال بالضبط إجابتين صحيحتين
p_x_eq_2 = binom_pmf(2, n, p)

# احتمال أن تكون كل الإجابات صحيحة (3 من 3)
p_x_eq_3 = binom_pmf(3, n, p)

# احتمال على الأقل إجابتين صحيحتين = P(2) + P(3)
p_x_ge_2 = p_x_eq_2 + p_x_eq_3

# احتمال على الأكثر إجابتين صحيحتين = 1 - P(3)
p_x_le_2 = 1 - p_x_eq_3

# المتوسط في Binomial: E[X] = n * p
expected_x = n * p

# التباين: Var(X) = n * p * (1 - p)
var_x = n * p * (1 - p)

# الانحراف المعياري = sqrt(Var)
std_x = math.sqrt(var_x)

print("\n=== Binomial MCQ Example ===")
print("P(X = 2)       =", p_x_eq_2)
print("P(X >= 2)      =", p_x_ge_2)
print("P(X <= 2)      =", p_x_le_2)
print("E[X]           =", expected_x)
print("Std dev(X)     =", std_x)


## الخلية 5 – مثال Normal (عدد المشاهدين لبرنامج TV)

In [ ]:
# معطيات التوزيع الطبيعي للمشاهدين (بالملايين)
mu = 1.5      # المتوسط 1.5 مليون
sigma = 0.2   # الانحراف المعياري 0.2 مليون

# 1) P(1.6 <= X <= 2) = F(2) - F(1.6)
p_16_to_2 = normal_cdf(2.0, mu, sigma) - normal_cdf(1.6, mu, sigma)

# 2) P(X >= 1) = 1 - F(1)
p_at_least_1m = 1 - normal_cdf(1.0, mu, sigma)

# 3) P(X > 2) = 1 - F(2)
p_exceed_2m = 1 - normal_cdf(2.0, mu, sigma)

print("\n=== Normal TV Viewers Example ===")
print("P(1.6 <= X <= 2.0) =", p_16_to_2)
print("P(X >= 1.0)        =", p_at_least_1m)
print("P(X > 2.0)         =", p_exceed_2m)

# (اختياري) رسم منحنى PDF للتوزيع الطبيعي
x_vals = np.linspace(0.8, 2.2, 400)  # مجال X للرسم
pdf_vals = (1 / (sigma * math.sqrt(2 * math.pi))) * np.exp(
    -((x_vals - mu) ** 2) / (2 * sigma ** 2)
)

plt.figure()
plt.plot(x_vals, pdf_vals)
plt.title("Normal PDF for TV viewers (millions)")
plt.xlabel("Viewers (millions)")
plt.ylabel("Density")
plt.grid(True, alpha=0.3)
plt.show()


## الخلية 6 – CLT Example: الأرقام 0..9 والمتوسطات

In [ ]:
# ننشئ مجموعة الأرقام من 0 إلى 9
D = np.arange(10)  # [0, 1, 2, ..., 9]

# -------- الخطوة 1: سحب 200 رقم من D مع الإرجاع --------

# نختار 200 رقم عشوائي من D
sample_200 = rng.choice(D, size=200, replace=True)

plt.figure()
# بنستخدم حدود bins بحيث يكون كل عمود centered على الأعداد الصحيحة
plt.hist(sample_200, bins=np.arange(-0.5, 10.5, 1), edgecolor='black')
plt.title("Histogram of 200 digits sampled from {0..9}")
plt.xlabel("Digit")
plt.ylabel("Count")
plt.grid(True, alpha=0.3)
plt.show()

# -------- الخطوة 2 + 3: متوسطات عينات حجمها 5 --------

means = []  # قائمة لحفظ المتوسطات

# نكرر عملية: اختيار 5 أرقام وحساب المتوسط، 200 مرة
for _ in range(200):
    five_digits = rng.choice(D, size=5, replace=True)
    means.append(five_digits.mean())

# نحولها لمصفوفة NumPy لسهولة التعامل
means = np.array(means)

plt.figure()
# نرسم هيستوغرام للمتوسطات مع density=True (يشبه PDF)
plt.hist(means, bins=15, edgecolor='black', density=True)
plt.title("Histogram of sample means (n=5, 200 samples)")
plt.xlabel("Sample mean")
plt.ylabel("Density")
plt.grid(True, alpha=0.3)
plt.show()

print("\n=== CLT Example (digits 0..9) ===")
print("Population mean (0..9)  :", D.mean())       # المتوسط النظري 4.5
print("Mean of sample means    :", means.mean())   # يجب أن يكون قريباً من 4.5
print("Std dev of sample means :", means.std())


## الخلية 7 – Gaussian MLE: تقدير μ و σ² من البيانات

In [ ]:
def gaussian_mle(x):
    """
    Maximum Likelihood Estimation for μ and σ² of a 1D Gaussian.

    المدخل:
        x : مصفوفة 1D من المشاهدات الحقيقية

    المخرجات:
        mu_hat    : تقدير μ (متوسط التوزيع)
        sigma2_hat: تقدير σ² (تباين التوزيع)

    الفكرة:
    - نفترض أن X_i ~ N(μ, σ²) i.i.d.
    - باستخدام MLE، الحل التحليلي يعطينا:
        μ̂ = (1/N) * Σ x_i
        σ̂² = (1/N) * Σ (x_i - μ̂)²
    - نستخدم القسمة على N (وليس N-1) لأن هذا تقدير Maximum Likelihood.
    """
    x = np.asarray(x)   # نتأكد أنها مصفوفة NumPy
    N = x.size          # عدد المشاهدات

    mu_hat = x.mean()   # متوسط العينة = تقدير μ̂
    sigma2_hat = np.sum((x - mu_hat) ** 2) / N  # تقدير σ̂² حسب MLE

    return mu_hat, sigma2_hat


# مثال تجريبي صغير
data = np.array([1.2, 0.9, 1.5, 1.3, 1.1])

mu_hat, sigma2_hat = gaussian_mle(data)

print("\n=== Gaussian MLE Example ===")
print("Data        :", data)
print("MLE mu      :", mu_hat)
print("MLE sigma^2 :", sigma2_hat)
print("MLE sigma   :", math.sqrt(sigma2_hat))
